# 05 — Feature engineering

Same model (Ridge): minimal features vs rich features (lags, rolling mean, day-of-week, month, days to next holiday). Show metric improvement and coefficient magnitude.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
from src.evaluation import train_val_split, mae, mape
data_path = root / "data" / "surgeries.csv"
if not data_path.exists():
    from src.data_generator import generate_surgery_counts
    generate_surgery_counts(output_path=str(data_path))
df = pd.read_csv(data_path, parse_dates=["date"])
series = df[(df["specialty_name"] == "General Surgery") & (df["hospital_id"] == "H1")][["date", "surgery_count"]].copy()
series = series.sort_values("date").reset_index(drop=True)
train, val = train_val_split(series, date_col="date", val_days_or_ratio=0.2)
n_val = len(val)
cut = val["date"].min()

## Build feature sets: minimal vs rich

In [ ]:
full = pd.concat([train, val], ignore_index=True)
# Minimal: lag_1 only
full["lag_1"] = full["surgery_count"].shift(1)
# Rich: lags 1,7,14; 7-day rolling mean; dow; month; days to next holiday (simple proxy)
for L in [7, 14]:
    full[f"lag_{L}"] = full["surgery_count"].shift(L)
full["roll7"] = full["surgery_count"].shift(1).rolling(7, min_periods=1).mean()
full["dow"] = full["date"].dt.dayofweek
full["month"] = full["date"].dt.month
# Days to next holiday: use a few fixed dates per year as proxy
holidays = pd.to_datetime([f"{y}-01-01" for y in range(2020,2025)] + [f"{y}-12-25" for y in range(2020,2025)])
def days_to_next_holiday(dates):
    out = np.full(len(dates), np.nan)
    for i, d in enumerate(dates):
        future = holidays[holidays >= d]
        out[i] = (future.iloc[0] - d).days if len(future) else 365
    return out
full["days_to_holiday"] = days_to_next_holiday(full["date"])
full = full.dropna()
feat_min = ["lag_1"]
feat_rich = ["lag_1", "lag_7", "lag_14", "roll7", "dow", "month", "days_to_holiday"]

## Compare MAE / MAPE: minimal vs rich

In [ ]:
from sklearn.linear_model import Ridge
X_train_min = full.loc[full["date"] < cut, feat_min]
y_train_ml = full.loc[full["date"] < cut, "surgery_count"]
X_val_min = full.loc[full["date"] >= cut, feat_min].head(n_val)
X_train_rich = full.loc[full["date"] < cut, feat_rich]
X_val_rich = full.loc[full["date"] >= cut, feat_rich].head(n_val)
y_val_arr = val["surgery_count"].values
pred_min = Ridge(alpha=1.0).fit(X_train_min, y_train_ml).predict(X_val_min)
pred_rich = Ridge(alpha=1.0).fit(X_train_rich, y_train_ml).predict(X_val_rich)
if len(pred_min) < n_val:
    pred_min = np.r_[pred_min, np.full(n_val - len(pred_min), pred_min[-1])]
if len(pred_rich) < n_val:
    pred_rich = np.r_[pred_rich, np.full(n_val - len(pred_rich), pred_rich[-1])]
print("Minimal (lag_1 only):  MAE =", round(mae(y_val_arr, pred_min), 2), " MAPE =", round(mape(y_val_arr, pred_min, skip_zeros=True), 1), "%")
print("Rich (lags+roll+dow+month+holiday): MAE =", round(mae(y_val_arr, pred_rich), 2), " MAPE =", round(mape(y_val_arr, pred_rich, skip_zeros=True), 1), "%")

## Feature importance (Ridge coefficients)

In [ ]:
model = Ridge(alpha=1.0).fit(X_train_rich, y_train_ml)
coef = pd.Series(model.coef_, index=feat_rich)
coef = coef.reindex(coef.abs().sort_values(ascending=False).index)
print(coef.to_string())